# 03 · S7 corrections and final verification (amendment 1.3)

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

Reruns **only** the steps that amendment 1.3 affects. No dataset is preprocessed again and no model is trained.
1. External targets (CSE-CIC-IDS2018, UNSW-NB15): the S7 records now compare with CICIDS2017 **train + val**
   (val sets the decision threshold, so it counts as tuning data). The exact train-only overlap stays as a
   descriptive figure.
2. Within-dataset S7 records are rebuilt so that each one carries its definition, the outputs it came from and
   its checksum. Their content should not change: the summary shows `reproduces previous = True` for each.
3. Verification now also checks every S7 sidecar and record, in a separate table from the primary data.

Run every cell **from top to bottom**, one at a time. Nothing to add or edit.
**Before you start:** *Runtime → Change runtime type → High-RAM*.

| Step | What it does | Time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `39 passed`) | 3 min |
| 2 | external-target overlap and S7 records (CICIDS2017 train + val) | 45–90 min |
| 3 | within-dataset S7 records for CICIDS2017, CICIoMT2024, ToN_IoT, CIC-IoT2023, NSL-KDD | 30–60 min |
| 4 | verify primary data and S7 artifacts (expected last line `all outputs verified`) | 20–40 min |
| 5 | summary tables | 1 min |

Each step saves to Drive and pushes to GitHub before the next starts. **If Colab disconnects:** run step 1 again,
then continue from the step that did not finish.

**Where results go:** `results/data_audit/cross_dataset_overlap.json`, `results/data_audit/<dataset>/near_duplicates.json`,
`results/data_audit/auxiliary_manifest.json` and `results/data_audit/verification.md`, in Drive at
`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/` and on GitHub; the S7 sidecars in
`MyDrive/PhD-AI-Cyber/data/splits/<dataset>/near*.parquet` (Drive only, checksums on GitHub).


## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `39 passed`)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pathlib, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

import datetime as _dt, os, pathlib, shutil, subprocess

RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')

def git(*args, check=False):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN or '\0', '***').strip()
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def sync(message=None):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    """
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    git('rebase', '--abort'); git('merge', '--abort')      # clear any half-finished earlier pull
    for attempt in range(3):
        git('fetch', REMOTE, 'main', check=True)
        base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
        changed = set(git('diff', '--name-only', base)[1].split())
        untracked = set(git('ls-files', '--others', '--exclude-standard')[1].split())
        # untracked non-record files survive the reset unless GitHub has a file at the same path
        untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
        changed |= untracked
        keep = sorted(f for f in changed if f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        other = sorted(f for f in changed if not f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        stamp = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        hold = pathlib.Path('/content/sync_hold') / stamp if pathlib.Path('/content').exists() else pathlib.Path(REPO_DIR).parent / f'.sync_hold_{stamp}'
        for f in keep:
            (hold / f).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
        if other:
            bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
            for f in other:
                (bak / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
            print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
        git('reset', '--hard', 'FETCH_HEAD', check=True)
        for f in keep:
            dest = pathlib.Path(REPO_DIR, f)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(hold / f, dest)
        if keep:
            git('add', '--', *keep, check=True)
            if git('diff', '--cached', '--quiet')[0] != 0:
                git('commit', '-m', message or 'Save results produced in Colab', check=True)
        code, out = git('push', REMOTE, 'HEAD:main')
        if code == 0:
            head = git('log', '--oneline', '-1')[1]
            print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
            return
        print('GitHub moved while saving; trying again')
    raise SystemExit('could not save to GitHub after 3 tries: copy the messages above into the project thread')


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

def push(message):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records."""
    sync(message)
    print('saved to GitHub:', message)

def run(*args, message):
    """Run one preprocessing command (its output shows below); push the results only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/preprocess.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')


## Step 2 · External targets: exact overlap (descriptive) and S7 records against CICIDS2017 train + val

In [ ]:
run('--cross-overlap', message='Cross-dataset overlap and S7 records vs CICIDS2017 train + val (amendment 1.3)')

## Step 3 · Within-dataset S7 records, with definition, provenance and checksum

Stops with an error if the recomputed near-duplicates do not match the preprocessing reports exactly.

In [ ]:
run('--near-duplicates', message='Within-dataset S7 records with provenance and checksums (amendment 1.3)')

## Step 4 · Verify primary data and S7 artifacts

In [ ]:
run('--verify', message='Verify primary data and S7 artifacts (amendment 1.3)')

## Step 5 · Summary

Expected: `reproduces previous` is True for the five within-dataset records; CICIDS2017 and NSL-KDD show `estimable False` (reported as not estimable, the rule is not changed); both verification tables show `yes` everywhere. Then send the finished notebook to the project thread.

In [ ]:
import json, glob
import pandas as pd
from IPython.display import Markdown, display
rows = []
for f in sorted(glob.glob('results/data_audit/*/near_duplicates.json')):
    s = json.load(open(f))
    for sig in ('sig3', 'sig2'):
        t = s['subsets'][sig].get('test', {})
        rows.append({'S7 record': s['dataset'], 'rounding': sig, 'test rows': t.get('rows'),
                     'unseen (kept in S7)': t.get('unseen_rows'), 'kept share': round(t.get('unseen_share') or 0, 4),
                     'attack blocks (all / kept)': f"{t.get('attack_blocks')} / {t.get('unseen_attack_blocks')}",
                     'benign blocks (all / kept)': f"{t.get('benign_blocks')} / {t.get('unseen_benign_blocks')}",
                     'estimable': t.get('estimable'), 'reproduces previous': s.get('reproduces_previous_sidecar')})
x = json.load(open('results/data_audit/cross_dataset_overlap.json'))
for d in ('csecicids2018_improved', 'unsw_nb15'):
    for sig in ('sig3', 'sig2'):
        t = x[d]['near_duplicates'][sig]
        rows.append({'S7 record': d + ' vs CICIDS2017 train + val', 'rounding': sig, 'test rows': t['rows'],
                     'unseen (kept in S7)': t['unseen_rows'], 'kept share': round(t['unseen_share'], 4),
                     'attack blocks (all / kept)': f"{t['attack_blocks']} / {t['unseen_attack_blocks']}",
                     'benign blocks (all / kept)': f"{t['benign_blocks']} / {t['unseen_benign_blocks']}",
                     'estimable': t['estimable'],
                     'reproduces previous': 'n/a (definition changed in 1.3)'})
display(pd.DataFrame(rows))
display(Markdown(open('results/data_audit/verification.md').read()))